In [67]:
using Plots
using NaturalNeighbours
using LinearAlgebra
using StableRNGs
using DelaunayTriangulation

In [68]:
function read_custom_file(filepath::String)
    array1 = Float64[]
    array2 = Float64[]

    start_reading = false

    num_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    open(filepath, "r") do file
        for line in eachline(file)

            clean_line = strip(line)

            if startswith(clean_line, "+") || startswith(clean_line, "-")
                matches = [m.match for m in eachmatch(num_regex, clean_line)]

                if length(matches) >= 2
                    push!(array1, parse(Float64, matches[1]))
                    push!(array2, parse(Float64, matches[2]))
                end
            else
                continue
            end
        end
    end
    return array1, array2
end

read_custom_file (generic function with 1 method)

In [69]:
cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/2.S9272.PtCoCu.7/S9272-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/6.S9281.CoPt.10/S9281-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/4.S9283.CoPt.6/S9283-FORC-50-1500-5s"
H_read = Float64[]
M_read = Float64[]
H_read, M_read = read_custom_file(cale * ".txt")

([1926.149, 1966.079, 2006.295, 1845.812, 1900.373, 1954.284, 2006.217, 1765.163, 1819.753, 1873.858  …  1516.857, 1570.778, 1625.594, 1679.432, 1733.308, 1788.094, 1841.95, 1895.938, 1950.765, 2004.618], [0.0002315941, 0.0002317994, 0.0002326804, 0.0002304468, 0.0002313527, 0.0002317877, 0.0002337235, 0.0002295009, 0.0002304182, 0.0002303912  …  0.0002133032, 0.0002193848, 0.0002233787, 0.0002226158, 0.0002270869, 0.000226353, 0.0002305136, 0.0002307518, 0.0002289827, 0.0002341819])

In [70]:
## Normalize
normH = 1.0e3
normM = 1.0e-3
H_read = H_read ./ normH
M_read = M_read ./ normM
H_read, M_read

([1.926149, 1.966079, 2.006295, 1.845812, 1.900373, 1.9542840000000001, 2.006217, 1.765163, 1.819753, 1.873858  …  1.516857, 1.570778, 1.625594, 1.679432, 1.733308, 1.788094, 1.84195, 1.8959380000000001, 1.950765, 2.004618], [0.2315941, 0.23179940000000002, 0.2326804, 0.2304468, 0.2313527, 0.23178769999999999, 0.2337235, 0.2295009, 0.2304182, 0.2303912  …  0.21330319999999997, 0.2193848, 0.2233787, 0.2226158, 0.22708689999999998, 0.226353, 0.23051359999999999, 0.23075179999999998, 0.22898269999999998, 0.2341819])

In [71]:
scatter(H_read, M_read,
    title="Scatter Plot of Data",
    xlabel="H",
    ylabel="M",
    legend=false,
    markersize=3,
    markercolor=:indianred,
    markerstrokecolor=:indianred
)

In [72]:
## Detect FORCs (Hr, numPointsPerFORC)
Hr_read = Float64[]
numPointsPerFORC = Int[]

current_Hr = H_read[1]
push!(Hr_read, current_Hr)

counterPointsPerFORC = 1

for i in 2:(length(H_read))
    if H_read[i] < H_read[i-1]
        global current_Hr = H_read[i]
        push!(numPointsPerFORC, counterPointsPerFORC)
        global counterPointsPerFORC = 0
    end
    global counterPointsPerFORC += 1
    push!(Hr_read, current_Hr)
end
push!(numPointsPerFORC, counterPointsPerFORC)
numFORCs = length(numPointsPerFORC)

println("----------- Total $(length(numPointsPerFORC)) FORCs -----------")
println("---------  $(numPointsPerFORC[1]) first / $(numPointsPerFORC[numFORCs]) last ---------")

----------- Total 50 FORCs -----------
---------  3 first / 75 last ---------


In [73]:
## Save Hr-H-M original file
n = length(Hr_read)
if length(H_read) != n || length(M_read) != n
    error("Error: All three arrays must have the same length.")
end

open(cale * "_Hr-H-M_orig.dat", "w") do file
    for i in 1:n
        println(file, "$(Hr_read[i])  $(H_read[i])  $(M_read[i])")
    end
end

In [74]:
function gimmeOneFORC(theFORC::Int64)
    contorNumPoints = 0
    H_interp = Float64[]
    M_interp = Float64[]
    #detect indexes for $(theFORC)
    if theFORC > 1
        for i in 1:theFORC-1
            contorNumPoints += numPointsPerFORC[i]
        end
    else
        contorNumPoints = 0
    end

    #myHr = Hr_read[contorNumPoints]
    startPointOnFORC = contorNumPoints + 1
    push!(H_interp, H_read[startPointOnFORC])
    push!(M_interp, M_read[startPointOnFORC])
    contorNumPoints = 1
    while (Hr_read[startPointOnFORC+contorNumPoints-1] - Hr_read[startPointOnFORC+contorNumPoints]) < 1.0e-5 #eps - compare floats
        push!(H_interp, H_read[startPointOnFORC+contorNumPoints])
        push!(M_interp, M_read[startPointOnFORC+contorNumPoints])
        contorNumPoints += 1
        if (startPointOnFORC + contorNumPoints) > length(Hr_read)
            break
        end
    end
    H_interp, M_interp
end

gimmeOneFORC (generic function with 1 method)

In [75]:
#f = (x, y) -> 0.75 * exp(-((9 * x - 2)^2 + (9 * y - 2)^2) / 4) + 0.75 * exp(-(9 * x + 1)^2 / 49 - (9 * y + 1) / 10) + 0.5 * exp(-((9 * x - 7)^2 + (9 * y - 3)^2) / 4) - 0.2 * exp(-(9 * x - 4)^2 - (9 * y - 7)^2)
#f′ = (x, y) -> [(exp(-(9 * x - 4)^2 - (9 * y - 7)^2) * (162 * x - 72)) / 5 - (3 * exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4) * ((81 * x) / 2 - 9)) / 4 - (exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4) * ((81 * x) / 2 - 63 / 2)) / 2 - (3 * exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10) * ((162 * x) / 49 + 18 / 49)) / 4
#    (exp(-(9 * x - 4)^2 - (9 * y - 7)^2) * (162 * y - 126)) / 5 - (3 * exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4) * ((81 * y) / 2 - 9)) / 4 - (exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4) * ((81 * y) / 2 - 27 / 2)) / 2 - (27 * exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)) / 40]
#f′′ = (x, y) -> [(162*exp(-(9 * x - 4)^2 - (9 * y - 7)^2))/5-(243*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10))/98-(243*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4))/8-(81*exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4))/4+(3*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49)^2)/4+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)^2)/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)^2)/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)^2)/5 (27*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49))/40+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)*((81*y)/2-9))/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)*((81*y)/2-27/2))/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)*(162*y-126))/5
#    (27*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10)*((162*x)/49+18/49))/40+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*x)/2-9)*((81*y)/2-9))/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*x)/2-63/2)*((81*y)/2-27/2))/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*x-72)*(162*y-126))/5 (243*exp(-(9 * y) / 10 - (9 * x + 1)^2 / 49 - 1 / 10))/400+(162*exp(-(9 * x - 4)^2 - (9 * y - 7)^2))/5-(243*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4))/8-(81*exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4))/4+(3*exp(-(9 * x - 2)^2 / 4 - (9 * y - 2)^2 / 4)*((81*y)/2-9)^2)/4+(exp(-(9 * x - 7)^2 / 4 - (9 * y - 3)^2 / 4)*((81*y)/2-27/2)^2)/2-(exp(-(9 * x - 4)^2 - (9 * y - 7)^2)*(162*y-126)^2)/5]

In [76]:
#x = LinRange(0, 1, 100)
#y = LinRange(0, 1, 100)
#z = [f(x, y) for x in x, y in y]
#∇ = [f′(x, y) for x in x, y in y]
#∇₁ = first.(∇)
#∇₂ = last.(∇)
#H = [f′′(x, y) for x in x, y in y]
#H₁₂ = getindex.(H, 2)
#;

In [77]:
#rng = StableRNG(9199)
#x = rand(rng, 1000)
#y = rand(rng, 1000)
#z = f.(x, y)
#tri = triangulate([x'; y'])
#vorn = voronoi(tri)


In [78]:
tri = triangulate([H_read'; Hr_read'])
vorn = voronoi(tri)

Voronoi Tessellation.
    Number of generators: 1961
    Number of polygon vertices: 3791
    Number of polygons: 1961
    Weighted: false

In [79]:
#points = [x'; y']
#z = f.(x, y)
#tri = triangulate(points)
#∇g = generate_gradients(tri, z)
#;

In [80]:
points = [H_read'; Hr_read']
tri = triangulate(points)
∇g = generate_gradients(tri, M_read)
;

In [81]:
#∇gr, _ = generate_derivatives(tri, z; method=Direct())

In [82]:
#to_mat(H::NTuple{3,Float64}) = [H[1] H[3]; H[3] H[2]]

In [83]:
#_, Hg = generate_derivatives(tri, z)

In [84]:
#_, Hg = generate_derivatives(tri, M_read)
_, Hg = generate_derivatives(tri, M_read, use_cubic_terms=false)
#_, Hg = generate_derivatives(tri, M_read, method=Iterative())

([(0.007133884575644937, -0.004619668524588034), (0.014360809803020477, -0.014569727515239236), (0.031249591584690047, -0.0020033565450513135), (0.015037816813917608, -1.1238622289618283e-6), (0.013302539217485415, 0.0005761403780745687), (0.020781942849351524, 0.0019034031421757609), (0.06494479500821458, 0.004583872885626277), (0.01670963431863376, -0.0027009675169367446), (0.011122500414789939, 0.0004911479459127499), (0.007363351772682464, -0.0007268342471388547)  …  (0.08320394041237517, 0.020566906071886028), (0.07175718787812288, -0.010469411374497438), (0.04633593533530036, -0.009749428590402455), (0.03380621704039064, 0.019997008521784185), (0.03326949345904722, -0.011059563850220033), (0.03495826538155009, 0.018174192136626813), (0.02472595643854043, -0.013561372085305005), (0.011458470032230135, 0.0032852232302569414), (0.043752658898158936, 0.022601571635338703), (0.13844846951652906, 0.00201000211224407)], [(0.13697092534868682, -0.12640122298499185, -0.060715530933644214)

In [85]:
myH = getindex.(Hg, 3)

#heatmap(myH)
;

In [86]:
scatter(H_read, Hr_read, marker_z=myH)

In [87]:
open(cale * "_VoronoiFORC.dat", "w") do file
    for i in 1:n
        println(file, "$(H_read[i])  $(Hr_read[i])  $(-myH[i])")
    end
end